## Conduct LLM experiments using direct and structured programming.

### Install packages/libraries

In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

import time
import random

from tqdm.auto import tqdm
from openai import RateLimitError, APIConnectionError, APITimeoutError

from sklearn.metrics import brier_score_loss, accuracy_score

### Retrieve data

In [2]:
data_dir = Path("../data/processed")

train_df = pd.read_csv(data_dir / "train.csv")
cal_df = pd.read_csv(data_dir / "calibration.csv")
test_df = pd.read_csv(data_dir / "test.csv")

print(len(train_df), len(cal_df), len(test_df))

283 91 87


### Initialize the API

In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI()

### Test LLM Prediction

In [7]:
def create_direct_prompt(row):
    return f"""
You are predicting whether an anonymous basketball player
will exceed a scoring threshold in their next game.

Historical statistics:
- Last 5 games average points: {row['PTS_L5']:.2f}
- Last 10 games average points: {row['PTS_L10']:.2f}
- Last 5 games points standard deviation: {row['PTS_STD_L5']:.2f}
- Last 10 games points standard deviation: {row['PTS_STD_10']:.2f}
- Last 10 games average minutes: {row['MIN_L10']:.2f}
- Recent scoring trend: {row['PTS_TREND']:.2f}
- Last 10 games over rate relative to this line: {row['HIST_OVER_RATE_L10']:.2f}

Scoring threshold: {row['PROP_LINE']:.1f} points

What is the probability that this player scores
MORE than the scoring threshold in their next game?

Return only a decimal probability between 0 and 1.
"""

In [19]:
results = []

for i, (_, row) in enumerate(train_df.head(5).iterrows()):

    prompt = create_direct_prompt(row)

    response = client.responses.create(
        model="gpt-5-mini",
        instructions=(
            "Use only the supplied statistics. "
            "Return only a decimal probability between 0 and 1."
        ),
        input=prompt,
        reasoning={"effort": "minimal"},
        max_output_tokens=500
    )

    if response.status != "completed":
        raise RuntimeError(f"Incomplete response at observation {i}")

    probability = float(response.output_text.strip())

    if not 0 <= probability <= 1:
        raise ValueError(f"Invalid probability: {probability}")

    results.append({
        "observation": i,
        "prop_line": row["PROP_LINE"],
        "actual_points": row["PTS"],
        "actual_over": int(row["TARGET_OVER"]),
        "historical_probability": row["HIST_OVER_RATE_L10"],
        "llm_probability": probability
    })

results_df = pd.DataFrame(results)

display(results_df)

,observation,prop_line,actual_points,actual_over,historical_probability,llm_probability
0,0,33.5,35,1,0.3,0.300
1,1,29.5,19,0,0.6,0.550
2,2,33.5,28,0,0.3,0.300
3,3,26.5,55,1,0.4,0.620
4,4,28.5,26,0,0.5,0.377


### Generate LLM predictions for the full dataset.

In [26]:
MODEL = "gpt-5-mini"

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)


def generate_llm_predictions(data, split_name):

    output_path = results_dir / f"direct_{split_name}.csv"

    if output_path.exists():
        results_df = pd.read_csv(output_path)
    else:
        results_df = pd.DataFrame(
            columns=["row_id", "llm_probability"]
        )

    completed = set(results_df["row_id"].astype(int))

    for i, (_, row) in enumerate(tqdm(data.iterrows(), total=len(data))):

        if i in completed:
            continue

        prompt = create_direct_prompt(row)

        for attempt in range(6):

            try:
                response = client.responses.create(
                    model=MODEL,
                    instructions=(
                        "Use only the supplied statistics. "
                        "Return only a decimal probability between 0 and 1."
                    ),
                    input=prompt,
                    reasoning={"effort": "minimal"},
                    max_output_tokens=500
                )

                if response.status != "completed":
                    raise RuntimeError(f"Incomplete response at row {i}")

                probability = float(response.output_text.strip())

                if not 0 <= probability <= 1:
                    raise ValueError(
                        f"Invalid probability at row {i}: {probability}"
                    )

                break

            except RateLimitError as e:

                if "insufficient_quota" in str(e):
                    raise RuntimeError(
                        "API quota exhausted. Check your billing."
                    ) from e

                if attempt == 5:
                    raise

                wait_time = min(60, 2 ** attempt + random.uniform(0, 1))

                print(f"Rate limited. Waiting {wait_time:.1f} seconds...")
                time.sleep(wait_time)

            except (APIConnectionError, APITimeoutError):

                if attempt == 5:
                    raise

                wait_time = min(60, 2 ** attempt + random.uniform(0, 1))
                time.sleep(wait_time)

        new_result = pd.DataFrame([{
            "row_id": i,
            "llm_probability": probability
        }])

        results_df = pd.concat(
            [results_df, new_result],
            ignore_index=True
        )

        results_df.to_csv(output_path, index=False)

        time.sleep(2)

    return results_df.sort_values("row_id").reset_index(drop=True)

In [22]:
cal_predictions = generate_llm_predictions(
    cal_df,
    "calibration"
)

display(cal_predictions.head())
print("Predictions:", len(cal_predictions))

  0%|          | 0/91 [00:00<?, ?it/s]

C:\Users\16827\AppData\Local\Temp\ipykernel_5724\723171588.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat(


,row_id,llm_probability
0,0,0.580
1,1,0.420
2,2,0.377
3,3,0.640
4,4,0.480


Predictions: 91


In [28]:
test_predictions = generate_llm_predictions(
    test_df,
    "test"
)

display(test_predictions.head())
print("Predictions:", len(test_predictions))

  0%|          | 0/87 [00:00<?, ?it/s]

Rate limited. Waiting 1.2 seconds...
Rate limited. Waiting 1.8 seconds...
Rate limited. Waiting 1.1 seconds...
Rate limited. Waiting 2.7 seconds...
Rate limited. Waiting 4.6 seconds...
Rate limited. Waiting 1.7 seconds...
Rate limited. Waiting 1.7 seconds...
Rate limited. Waiting 1.6 seconds...
Rate limited. Waiting 1.5 seconds...
Rate limited. Waiting 1.4 seconds...
Rate limited. Waiting 2.7 seconds...
Rate limited. Waiting 4.7 seconds...
Rate limited. Waiting 8.5 seconds...
Rate limited. Waiting 16.7 seconds...
Rate limited. Waiting 1.8 seconds...
Rate limited. Waiting 2.3 seconds...
Rate limited. Waiting 4.6 seconds...
Rate limited. Waiting 1.2 seconds...
Rate limited. Waiting 2.5 seconds...
Rate limited. Waiting 1.2 seconds...
Rate limited. Waiting 1.7 seconds...
Rate limited. Waiting 1.4 seconds...
Rate limited. Waiting 2.3 seconds...
Rate limited. Waiting 1.7 seconds...
Rate limited. Waiting 1.9 seconds...
Rate limited. Waiting 1.6 seconds...
Rate limited. Waiting 1.2 seconds...


,row_id,llm_probability
0,0,0.45
1,1,0.58
2,2,0.64
3,3,0.36
4,4,0.57


Predictions: 87


In [29]:
results_dir = Path("../results")

cal_predictions = pd.read_csv(results_dir / "direct_calibration.csv")
test_predictions = pd.read_csv(results_dir / "direct_test.csv")

print(f"Calibration: {len(cal_predictions)}/{len(cal_df)}")
print(f"Test: {len(test_predictions)}/{len(test_df)}")

Calibration: 91/91
Test: 87/87


In [32]:
def attach_predictions(data, predictions):

    data = data.reset_index(drop=True).copy()
    data["row_id"] = np.arange(len(data))

    return data.merge(
        predictions,
        on="row_id",
        how="left",
        validate="one_to_one"
    )


cal_results = attach_predictions(cal_df, cal_predictions)
test_results = attach_predictions(test_df, test_predictions)

assert cal_results["llm_probability"].notna().all()
assert test_results["llm_probability"].notna().all()

display(test_results[
    [
        "PLAYER_NAME",
        "PROP_LINE",
        "PTS",
        "TARGET_OVER",
        "HIST_OVER_RATE_L10",
        "llm_probability"
    ]
].head())

,PLAYER_NAME,PROP_LINE,PTS,TARGET_OVER,HIST_OVER_RATE_L10,llm_probability
0,Nikola Jokic,29.5,24,0.0,0.6,0.45
1,Luka Doncic,33.5,30,0.0,0.4,0.58
2,Giannis Antetokounmpo,23.5,31,1.0,0.4,0.64
3,Donovan Mitchell,25.5,26,1.0,0.4,0.36
4,Jalen Brunson,23.5,30,1.0,0.7,0.57


In [34]:
def expected_calibration_error(y_true, y_prob, n_bins=5):

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(y_prob, bin_edges[1:-1], right=True)

    ece = 0.0

    for i in range(n_bins):

        mask = bin_ids == i

        if mask.sum() == 0:
            continue

        avg_confidence = y_prob[mask].mean()
        actual_frequency = y_true[mask].mean()

        ece += (
            mask.sum() / len(y_true)
        ) * abs(avg_confidence - actual_frequency)

    return ece


def evaluate_predictions(data, probability_column):

    y_true = data["TARGET_OVER"].astype(int)
    y_prob = data[probability_column].astype(float)

    return {
        "Brier Score": brier_score_loss(y_true, y_prob),
        "ECE": expected_calibration_error(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_prob >= 0.5)
    }

In [35]:
train_over_rate = train_df["TARGET_OVER"].mean()

test_results["constant_probability"] = train_over_rate

comparison = pd.DataFrame({
    "Historical Frequency": evaluate_predictions(
        test_results, "HIST_OVER_RATE_L10"
    ),
    "Constant Probability": evaluate_predictions(
        test_results, "constant_probability"
    ),
    "LLM Direct Prompting": evaluate_predictions(
        test_results, "llm_probability"
    )
}).T

display(comparison.round(4))

,Brier Score,ECE,Accuracy
Historical Frequency,0.2539,0.0563,0.5172
Constant Probability,0.2538,0.0738,0.4598
LLM Direct Prompting,0.2444,0.0970,0.5632


In [36]:
comparison.to_csv(results_dir / "direct_comparison.csv")

cal_results.to_csv(
    results_dir / "direct_calibration_full.csv",
    index=False
)

test_results.to_csv(
    results_dir / "direct_test_full.csv",
    index=False
)